# PowerGuard AI: обучение двух детекторов ЛЭП

Продуктовый трек. Две отдельные ячейки обучения: YOLO11n (быстрая) и YOLO11s (точная). Перед запуском выберите GPU runtime и укажите путь к `data.yaml`.

## Метрики

Основная метрика — mAP@0.5: она задана критерием проекта и суммирует precision–recall по классам. Дополнительно сравниваем mAP@0.5:0.95 (строгость локализации), precision, recall, F1 и latency. Для инспекции особенно важен recall, поскольку пропущенный дефект потенциально дороже ложной тревоги; рабочий порог затем выбирается по F1 и бизнес-риску.

In [ ]:
%pip install -q "ultralytics>=8.3,<9" pandas matplotlib seaborn pyyaml

In [ ]:
from pathlib import Path
import random, shutil, time, yaml
import pandas as pd
from ultralytics import YOLO

SEED = 42
DATA_YAML = Path('/content/data/lep/data.yaml')  # измените путь
RUNS = Path('/content/runs_powerguard')
EXPORT = Path('/content/exported_models')
EXPORT.mkdir(parents=True, exist_ok=True)
assert DATA_YAML.is_file(), f'Не найден {DATA_YAML}'
print('Dataset:', DATA_YAML)

## Аудит датасета

В обсуждении проекта у части участников архив содержал около 5 042 изображений вместо заявленных 7 988. Эта ячейка фиксирует фактический размер, проверяет пары изображение/label и распределение классов.

In [ ]:
cfg = yaml.safe_load(DATA_YAML.read_text(encoding='utf-8'))
root = Path(cfg.get('path', DATA_YAML.parent))
if not root.is_absolute(): root = DATA_YAML.parent / root
names = cfg['names']
names = names if isinstance(names, list) else [names[i] for i in sorted(names)]
rows, missing = [], []
for split in ('train', 'val', 'test'):
    image_dir = root / cfg.get(split, f'images/{split}')
    label_dir = Path(str(image_dir).replace('images', 'labels'))
    images = [p for p in image_dir.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}] if image_dir.exists() else []
    for image in images:
        label = label_dir / image.relative_to(image_dir).with_suffix('.txt')
        if not label.exists(): missing.append(str(image))
        elif label.stat().st_size:
            for line in label.read_text().splitlines(): rows.append((split, int(line.split()[0])))
    print(f'{split}: {len(images)} images')
print('Images without labels:', len(missing))
dist = pd.DataFrame(rows, columns=['split','class_id'])
dist['class_name'] = dist.class_id.map(dict(enumerate(names)))
display(pd.crosstab(dist.class_name, dist.split).fillna(0).astype(int))

## Модель 1 — YOLO11n (быстрый профиль)

Отдельная ячейка запуска обучения №1. `pretrained=True` соответствует продуктовому треку: готовая архитектура дообучается на целевом датасете.

In [ ]:
fast_model = YOLO('yolo11n.pt')
fast_train = fast_model.train(
    data=str(DATA_YAML), epochs=80, imgsz=640, batch=-1,
    optimizer='AdamW', lr0=0.001, cos_lr=True, patience=20,
    degrees=5, translate=0.1, scale=0.35, fliplr=0.5, mosaic=0.7,
    seed=SEED, deterministic=True, project=str(RUNS), name='yolo11n_lep',
)

## Модель 2 — YOLO11s (точный профиль)

Отдельная ячейка запуска обучения №2. Большее разрешение помогает мелким объектам, но увеличивает latency и расход памяти.

In [ ]:
accurate_model = YOLO('yolo11s.pt')
accurate_train = accurate_model.train(
    data=str(DATA_YAML), epochs=100, imgsz=960, batch=-1,
    optimizer='AdamW', lr0=0.0008, cos_lr=True, patience=25,
    degrees=5, translate=0.1, scale=0.4, fliplr=0.5, mosaic=0.8, close_mosaic=10,
    seed=SEED, deterministic=True, project=str(RUNS), name='yolo11s_lep',
)

## Честное сравнение на одном validation split

После обоих запусков валидируем лучшие checkpoints. Для latency делаем warm-up и несколько повторов на одном устройстве.

In [ ]:
experiments = [
    ('fast', RUNS/'yolo11n_lep/weights/best.pt', 640),
    ('accurate', RUNS/'yolo11s_lep/weights/best.pt', 960),
]
comparison = []
for profile, weights, imgsz in experiments:
    model = YOLO(str(weights))
    metrics = model.val(data=str(DATA_YAML), split='val', imgsz=imgsz, plots=True)
    comparison.append({
        'profile': profile, 'imgsz': imgsz,
        'mAP50': metrics.box.map50, 'mAP50-95': metrics.box.map,
        'precision': metrics.box.mp, 'recall': metrics.box.mr,
        'inference_ms': metrics.speed['inference'],
    })
comparison = pd.DataFrame(comparison).sort_values('mAP50', ascending=False)
display(comparison.style.format({c:'{:.4f}' for c in comparison.columns if c not in ('profile','imgsz')}))

In [ ]:
shutil.copy2(RUNS/'yolo11n_lep/weights/best.pt', EXPORT/'yolo11n_lep.pt')
shutil.copy2(RUNS/'yolo11s_lep/weights/best.pt', EXPORT/'yolo11s_lep.pt')
comparison.to_csv(EXPORT/'model_comparison.csv', index=False)
print('Экспортировано:', list(EXPORT.iterdir()))

## Анализ и гипотезы (заполнить фактическими результатами)

Приложенный лучший запуск дал mAP@0.5 = 0.902. Наиболее сильные классы: `bad_insulator` (0.979), `safety_sign+` (0.977), `nest` (0.948); слабые: `polymer_insulators` (0.771), `vibration_damper` (0.792). Возможные причины — мелкий масштаб, дисбаланс, сложный фон и неоднозначная разметка.

После выполнения notebook вставьте сюда: (1) какая из двух моделей обучилась быстрее; (2) разницу mAP и latency; (3) признаки переобучения по графикам; (4) примеры false positive/false negative; (5) решение, какая модель является быстрой и какая точной по фактическим данным. Предлагаемые улучшения: class-aware sampling, SAHI/tiling, больше примеров слабых классов, hard-negative mining и проверка утечек между train/val.